In [0]:
#------
# 1. Start of Gold layer
#  ------

from pyspark.sql import functions as F

df_silver = spark.table("workspace.silver.sales_enriched")

df_revenue = df_silver.filter(F.col("is_revenue_recognized") == True)

print("Silver rows:", df_silver.count(), "| Revenue-recognized rows:", df_revenue.count())



In [0]:
#-------
# 2.Daily sales summary
#-------------

df_daily_summary = (
    df_revenue
    .groupBy(F.to_date(F.col("order_date")).alias("order_date") , "category")
    .agg(
        F.count(F.col("order_id")).alias("total_orders"),
        F.sum(F.col("quantity")).alias("total_quantity"),
        F.round(F.sum(F.col("gross_amount")),2).alias("gross_revenue"),  
        F.round(F.sum(F.col("profit_amount")),2).alias("total_profit"),
    )
    .orderBy(F.col("order_date"), F.col("category"))
)

display(df_daily_summary.limit(100))
#-------
#

df_daily_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.daily_sales_summary")

In [0]:
#-------------------------------
# 3. TOP CUSTOMERS (lifetime value)
#------------------------------

df_top_customer = (

    df_revenue
    .groupBy(F.col("customer_id"),  F.col("region"), F.col("customer_segment"))
    .agg(
        F.count(F.col("order_id")).alias("total_orders"),
        F.round(F.sum(F.col("gross_amount")),2).alias("lifetime_revenue"),
    )
    .orderBy(F.col("lifetime_revenue").desc())
    
)

display(df_top_customer.limit(20))
#-------
#
df_top_customer.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.top_customers")

In [0]:
#-------------------------------
# 4. Monthly Category Performance
#------------------------------

df_monthly_category_performance = (
    df_revenue
    .groupBy(F.year(F.col("order_date")).alias("order_year"), F.month(F.col("order_date")).alias("order_month"), F.col("category"))
    .agg(
        F.round(F.sum(F.col("gross_amount")),2).alias("gross_revenue"),
        F.round(F.sum(F.col("profit_amount")),2).alias("total_profit"),
    )
    .withColumn("margin_pct", F.round((F.col("total_profit")/F.col("gross_revenue"))*100,2))
    .orderBy(F.col("order_year"), F.col("order_month"), F.col("category"))
)

display(df_monthly_category_performance.limit(100))
#-------
#
(df_monthly_category_performance
 .write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("workspace.gold.monthly_category_performance")
 )